## 01 — Measuring the Problem

We know the structure of the data. Now we measure it.

A 40 MB GeoJSON file loaded on every map interaction would be unusable in a real application. But "40 MB feels slow" is not a precise enough reason to build a whole LOD pipeline. We want numbers.

This notebook answers:

1. How large is the file on disk?
2. How many total coordinate points are we holding in memory?
3. How long does a load-and-parse cycle take?
4. What does it look like to display a sample on a map?

## Setup

Load the file and import timing tools.

In [ ]:
import json
import time
from pathlib import Path

data_path = Path("ne_10m_railroads.geojson")

with open(data_path) as f:
    railroads = json.load(f)

features = railroads["features"]
print(f"Loaded {len(features):,} features")

Loaded 25,413 features


## File Size on Disk

In [ ]:
size_bytes = data_path.stat().st_size
size_mb = size_bytes / 1_000_000

print(f"File size: {size_bytes:,} bytes")
print(f"File size: {size_mb:.2f} MB")

File size: 39,598,080 bytes
File size: 39.60 MB


## Total Coordinate Points

File size tells us storage cost. Coordinate count tells us rendering cost — each point must be projected and drawn.

Let's count the total number of `[lon, lat]` pairs across every feature.

In [ ]:
total_coords = sum(len(f["geometry"]["coordinates"]) for f in features)

avg_coords = total_coords / len(features)

print(f"Total coordinate pairs: {total_coords:,}")
print(f"Average per feature:    {avg_coords:.1f}")

Total coordinate pairs: 1,396,480
Average per feature:    55.0


## Load Time

How long does it take to open and parse the file from scratch?

We time just the load — no display, no processing.

In [ ]:
start = time.perf_counter()

with open(data_path) as f:
    _ = json.load(f)

elapsed = time.perf_counter() - start

print(f"Load time: {elapsed:.3f} seconds")

Load time: 1.130 seconds


One to three seconds on most machines — just to parse the file, before any rendering.

A map that takes 2 seconds to respond to every pan or zoom is not usable.

## Coordinate Distribution

Not all features are equal. Some segments are short (a few points), some are long (hundreds of points).

Let's see the distribution.

In [ ]:
coord_counts = [len(f["geometry"]["coordinates"]) for f in features]

coord_counts.sort()

n = len(coord_counts)
print(f"Min points in a feature:    {coord_counts[0]}")
print(f"Median points per feature:  {coord_counts[n // 2]}")
print(f"90th percentile:            {coord_counts[int(n * 0.9)]}")
print(f"Max points in a feature:    {coord_counts[-1]}")

Min points in a feature:    2
Median points per feature:  24
90th percentile:            136
Max points in a feature:    500


## Displaying a Sample on a Map

Displaying all 25,000+ features at once would freeze the notebook. Instead, we display a **random sample of 500 features** — enough to see what the data looks like, not enough to crash the browser.

This itself is a preview of what we will build: a system that only shows you the data that is appropriate for your current view.

In [ ]:
import random
from ipyleaflet import Map, GeoJSON

random.seed(42)
sample = random.sample(features, 500)

sample_collection = {
    "type": "FeatureCollection",
    "features": sample
}

m = Map(center=[20, 0], zoom=2)

layer = GeoJSON(
    data=sample_collection,
    style={"color": "#cc3300", "weight": 1, "opacity": 0.7}
)

m.add(layer)
m

Map(center=[20, 0], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_out_text…

Even 500 features gives a reasonable picture of global coverage. Now imagine loading all 25,413 every time you pan the map.

## Summarizing the Problem

Let's put the numbers in one place.

In [ ]:
print("=" * 40)
print("Railroad Dataset — Problem Summary")
print("=" * 40)
print(f"  File size:          {size_mb:.1f} MB")
print(f"  Features:           {len(features):,}")
print(f"  Total coordinates:  {total_coords:,}")
print(f"  Avg coords/feature: {avg_coords:.1f}")
print(f"  Load time:          ~{elapsed:.1f}s")
print("=" * 40)
print("Every pan/zoom at this cost = unusable map")

Railroad Dataset — Problem Summary
  File size:          39.6 MB
  Features:           25,413
  Total coordinates:  1,396,480
  Avg coords/feature: 55.0
  Load time:          ~0.2s
Every pan/zoom at this cost = unusable map


## Exercise A

Find the **10 features with the most coordinate points**. For each, print the feature index (its position in the features list) and its coordinate count.

Which feature has the most points? What are its properties?

In [ ]:
# Find the 10 features with the most coordinate points
# Print index and coordinate count for each
# Your code here

In [ ]:
# Find the 10 features with the most coordinate points
coord_counts = [
    (i, len(f["geometry"]["coordinates"]))
    for i, f in enumerate(features)
]

# Sort by coordinate count (descending)
coord_counts.sort(key=lambda x: x[1], reverse=True)

top_10 = coord_counts[:10]

for idx, count in top_10:
    print(f"Feature index: {idx}, coordinate count: {count}")

# Identify the largest feature
largest_idx, largest_count = top_10[0]
largest_feature = features[largest_idx]

print("\nLargest feature properties:")
print(largest_feature.get("properties", {}))

Feature index: 146, coordinate count: 500
Feature index: 243, coordinate count: 500
Feature index: 529, coordinate count: 500
Feature index: 966, coordinate count: 500
Feature index: 1392, coordinate count: 500
Feature index: 1936, coordinate count: 500
Feature index: 3338, coordinate count: 500
Feature index: 3377, coordinate count: 500
Feature index: 3429, coordinate count: 500
Feature index: 3458, coordinate count: 500

Largest feature properties:
{'rwdb_rr_id': 147, 'mult_track': 1, 'electric': 1, 'other_code': 1, 'category': 2, 'disp_scale': '1:40m', 'add': 0, 'featurecla': 'Railroad', 'scalerank': 6, 'natlscale': 40, 'part': 'ne_global_not_north_america', 'continent': 'Europe'}


## Exercise B

The `scalerank` property controls at what zoom level a feature should appear. Features with `scalerank <= 3` are the most important — major trunk lines.

1. Count how many features have `scalerank <= 3`
2. Sum their coordinate points
3. What percentage of total coordinates do these "important" features account for?

In [ ]:
# Count high-importance features (scalerank <= 3) and their coordinate share
# Your code here

In [ ]:
# 1. Filter important features
important_features = [
    f for f in features
    if f["properties"].get("scalerank", 999) <= 3
]

# 2. Count coordinate points for important features
important_coords = sum(
    len(f["geometry"]["coordinates"])
    for f in important_features
)

# 3. Compute totals
total_coords = sum(
    len(f["geometry"]["coordinates"])
    for f in features
)

# 4. Percentage share
percent = (important_coords / total_coords) * 100

print(f"Important features (scalerank <= 3): {len(important_features):,}")
print(f"Coordinates in important features:   {important_coords:,}")
print(f"Total coordinates:                   {total_coords:,}")
print(f"Percentage of total:                {percent:.2f}%")

Important features (scalerank <= 3): 0
Coordinates in important features:   0
Total coordinates:                   1,396,480
Percentage of total:                0.00%


## Check Your Understanding

We timed the **file load** at roughly 1–3 seconds. But loading is only part of the cost.

Name **two other operations** that happen between "file loaded" and "feature visible on the map" — operations that also take time and scale with feature count.

You do not need to write code. Write your answer as two bullet points.

---

Projection to screen space
Converting geographic coordinates (lon/lat) into pixel coordinates using the current map projection. This happens for every visible point on every pan/zoom.
Geometry clipping and rendering
Determining which parts of each feature fall inside the current viewport, then rasterizing/drawing them onto the map canvas .

## Next

In [Module 01 — Douglas-Peucker Simplification](../01-Douglas_Peucker/README.md), we build the algorithm that reduces point count while preserving line shape.